# Phase 4, Step 1: Latent Space Exploration & Shape Morphing
This notebook demonstrates how our trained 3D PointNet-VAE learns a continuous, smooth representation of vehicle geometries. We will:
1. Load the trained VAE model.
2. Identify the lowest-drag and highest-drag vehicles in our test set.
3. Encode both vehicles into their 128-dimensional latent vectors.
4. Perform **linear latent space interpolation** (morphing) between them.
5. Decode and plot the intermediate 3D shapes to observe the continuous geometric transition.

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Ensure project root is in the system path
sys.path.append(os.path.abspath(".."))
from src.dataset import VehiclePointCloudDataset
from src.models.vae import PointNetVAE

### 1. Identify Low-Drag and High-Drag Cars
We read the master metadata file to find the cars with the minimum and maximum drag area index (`drag_area`) in the test dataset.

In [ ]:
metadata = pd.read_csv("../metadata/metadata.csv")
test_cars = metadata[metadata["split"] == "test"]
print(f"Total cars in test set: {len(test_cars)}")

# Find lowest and highest drag cars
low_drag_row = test_cars.loc[test_cars["drag_area"].idxmin()]
high_drag_row = test_cars.loc[test_cars["drag_area"].idxmax()]

print(f"\nLow-Drag Car: ID={low_drag_row['id']}, Cd={low_drag_row['cd']:.4f}, Drag Area={low_drag_row['drag_area']:.4f}")
print(f"High-Drag Car: ID={high_drag_row['id']}, Cd={high_drag_row['cd']:.4f}, Drag Area={high_drag_row['drag_area']:.4f}")

### 2. Load the Dataset and Model
We instantiate the PyTorch Dataset (using `num_points=2048`) and load our trained VAE weights.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load full test dataset
dataset = VehiclePointCloudDataset(csv_path="../metadata/metadata.csv", scales_path="../metadata/target_scales.json", split="test", num_points=2048, normalize_targets=False)

# Instantiate and load VAE model
vae = PointNetVAE(in_channels=6, latent_dim=128, num_points=2048).to(device)
vae.load_state_dict(torch.load("../models/vae_best.pth", map_location=device))
vae.eval()
print("PointNet-VAE successfully loaded!")

### 3. Extract and Encode Target Cars
We locate the point clouds for our chosen low-drag and high-drag cars, load them, and feed them through the encoder to extract their latent representations ($\mu$).

In [ ]:
# Get dataset indices for the low and high drag cars
low_drag_idx = dataset.df[dataset.df["id"] == low_drag_row["id"]].index[0]
high_drag_idx = dataset.df[dataset.df["id"] == high_drag_row["id"]].index[0]

# Load features
low_features, _ = dataset[low_drag_idx]
high_features, _ = dataset[high_drag_idx]

# Add batch dimension and send to device: shape [1, 6, 2048]
low_features = low_features.unsqueeze(0).to(device)
high_features = high_features.unsqueeze(0).to(device)

# Encode shapes to get latent codes (using mean vector mu)
with torch.no_grad():
    mu_low, _ = vae.encoder(low_features)
    mu_high, _ = vae.encoder(high_features)
    
print(f"Encoded low-drag car to latent code of shape: {mu_low.shape}")
print(f"Encoded high-drag car to latent code of shape: {mu_high.shape}")

### 4. Latent Space Interpolation
We will perform **linear interpolation (LERP)** between the low-drag and high-drag shape codes:
$$ z(\alpha) = (1 - \alpha) \cdot \mu_{\text{high}} + \alpha \cdot \mu_{\text{low}} $$
where $\alpha$ ranges from $0$ (100% High-Drag) to $1$ (100% Low-Drag). We then decode these codes back to 3D point cloud coordinates.

In [ ]:
num_steps = 5
alphas = np.linspace(0.0, 1.0, num_steps)
reconstructed_clouds = []

with torch.no_grad():
    for alpha in alphas:
        # Interpolate between latent vectors
        z_interp = (1.0 - alpha) * mu_high + alpha * mu_low
        
        # Decode back to point cloud: shape [1, 3, 2048]
        recon_points = vae.decoder(z_interp)
        
        # Squeeze batch dimension and transpose to [2048, 3] for plotting
        points_np = recon_points.squeeze(0).transpose(0, 1).cpu().numpy()
        reconstructed_clouds.append(points_np)

print(f"Generated {len(reconstructed_clouds)} morphed shapes along the interpolation trajectory.")

### 5. Plot Morphed Car Shapes (3D Matplotlib Visualizations)
We plot the morphed point clouds side-by-side. You will see the physical geometry gradually adapt, morphing the details from the high-drag vehicle structure to the low-drag design profile.

In [ ]:
fig = plt.figure(figsize=(20, 6))
titles = [f"Alpha={a:.2f}\n({int((1-a)*100)}% High / {int(a*100)}% Low)" for a in alphas]

for idx, points in enumerate(reconstructed_clouds):
    ax = fig.add_subplot(1, num_steps, idx + 1, projection='3d')
    
    # Scatter plot: color points by depth (Z coord) to enhance 3D perception
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    sc = ax.scatter(x, y, z, c=z, cmap='viridis', s=1, alpha=0.6)
    
    ax.set_title(titles[idx], fontsize=12)
    
    # Keep coordinate limits consistent to see physical changes
    ax.set_xlim(-0.6, 0.6)
    ax.set_ylim(-0.6, 0.6)
    ax.set_zlim(-0.6, 0.6)
    
    # Set view angle (Elevation, Azimuth) to see the car's silhouette profile (side/isometric view)
    ax.view_init(elev=20, azim=-60)
    ax.axis('off') # Remove box lines for clean visuals

plt.suptitle("Generative 3D Shape Morphing (High-Drag to Low-Drag Car Body)", fontsize=16, y=0.98)
plt.tight_layout()

# Save figure
os.makedirs("../metadata", exist_ok=True)
plt.savefig("../metadata/vae_morphing_interpolation.png", dpi=300)
plt.show()